In [ ]:
# Import packages
import os
import re
import csv
import json
import shutil
import pandas as pd
import pyodbc
import yaml
import requests
import numpy as np
from dotenv import load_dotenv
from datetime import datetime
from time import time
from typing import Dict, List, Any
from msdev_kit import Auth
from msdev_kit.fabric import Workspace, Dataflow, Dataset, Report, Capacity, KQLDatabase, Operations
from msdev_kit.graph import GraphClient
from msdev_kit.sharepoint import SharePointClient

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

load_dotenv('./utils/.env')

# Paths
ROOT_PATH = './data/data_hub'
SHAREPOINT_ROOT_PATH = 'Shared Documents/General/2. Arquivos/8. Data Hub'

# Tenant/app settings
TENANT_ID = os.environ.get('TENANT_ID', '')
CLIENT_ID = os.environ.get('CLIENT_ID', '')
CLIENT_SECRET = os.environ.get('CLIENT_SECRET', '')
AZURE_SUBSCRIPTION_ID = os.environ.get('AZURE_SUBSCRIPTION_ID', '')
AZURE_RESOURCE_GROUP_ID = os.environ.get('AZURE_RESOURCE_GROUP_ID', '')

# Constant variable to see if we must erase current progress
ERASE_PROGRESS = False

In [ ]:
# PBI API Authentication (get bearer token)
fabric_auth = Auth(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
token = fabric_auth.get_token('pbi')
fabric_token = fabric_auth.get_token('fabric')
# pbi_user_token = fabric_auth.get_token_for_user('pbi')
# azure_user_token = fabric_auth.get_token_for_user('azure')
# outlook_user_token = fabric_auth.get_token_for_user('outlook')

In [ ]:
# Initializing objects
workspace = Workspace(token)
dataset = Dataset(token)
dataflow = Dataflow(token)
report = Report(token)
operations = Operations(fabric_token)
# pbi_capacities = Capacity(pbi_token=pbi_user_token)
# fabric_capacities = Capacity(azure_user_token)
kqldb = KQLDatabase("https://trd-s5p05dwxr9n10trm6q.z3.kusto.fabric.microsoft.com/", "Monitoring Eventhouse", CLIENT_ID, CLIENT_SECRET, TENANT_ID)

In [ ]:
df_workspaces = pd.DataFrame(workspace.list_workspaces().get('content', []))
print(f'Found {df_workspaces.index.size} workspaces:')
df_workspaces

In [ ]:
# df_capacities = pd.DataFrame(pbi_capacities.list_powerbi_capacities().get('content', []))
# df_capacities.head()

In [ ]:
# search_capacity = 'People BI - Capacity'
# desired_capacity_id = df_capacities.loc[df_capacities['displayName'] == search_capacity, 'id'].values[0]
# df_workspaces_in_capacity = df_workspaces.loc[df_workspaces['capacityId'] == desired_capacity_id]
# df_workspaces_in_capacity.reset_index(drop=True, inplace=True)
# df_workspaces_in_capacity.to_excel(f'./data/{search_capacity}_workspaces.xlsx', index=False)
# print(f'Found {len(df_workspaces_in_capacity)} workspaces in capacity "{search_capacity}".')
# df_workspaces_in_capacity

In [ ]:
# df_fabric_capacities = pd.DataFrame(fabric_capacities.list_fabric_capacities(azure_subscription_id=AZURE_SUBSCRIPTION_ID, azure_resource_group=AZURE_RESOURCE_GROUP_ID).get('content', []))
# print(f'Found {len(df_fabric_capacities)} fabric capacit{"y" if len(df_fabric_capacities) == 1 else "ies"}.')
# df_fabric_capacities.head()

In [ ]:
# # workspaces_to_change_capacity = [
# #     'e8f2f777-f2c5-416e-ab6a-fbff9eb512be',
# #     '20b2bae3-14ce-4a3c-bf24-8bbd3b16b3eb',
# #     '5d241e1c-064c-4ce0-842f-f851ef6bac6d'
# # ]
# workspaces_to_change_capacity = df_workspaces_in_capacity['id'].tolist()
# new_capacity = 'brewdatpeoplefabricp'
# new_capacity_id = df_capacities.loc[df_capacities['displayName'] == new_capacity, 'id'].values[0]

# for workspace_id in workspaces_to_change_capacity:
#     try:
#         response = pbi_capacities.assign_workspace_to_capacity(workspace_id, new_capacity_id)
#         print(f'Workspace {workspace_id} assigned to capacity {new_capacity} ({new_capacity_id}): {response}')
#     except Exception as e:
#         print(f'Failed to assign workspace {workspace_id} to capacity {new_capacity} ({new_capacity_id}): {e}')

In [ ]:
workspace_to_get_reports = 'Brewdat Prod - GHQ - People - PSI'

mask = df_workspaces['name'] == workspace_to_get_reports
workspace_id_series = df_workspaces.loc[mask, 'id']
if not workspace_id_series.empty:
    workspace_id = workspace_id_series.iloc[0]
else:
    workspace_id = None

df_reports  = pd.DataFrame(report.list_reports(workspace_id).get('content', []))
df_reports.head()

In [ ]:
# workspace_id = '837f9ce6-15e3-4e57-9b93-8bd9917a9579'
# report_id = 'afb30768-a6e4-4ffc-a566-b43feb6cb8d6'
# report_json = report.get_legacy_report_json(workspace_id=workspace_id, report_id=report_id, operations=operations).get('content')
# df_report_pages = report.get_legacy_report_pages_and_visuals(report_json, workspace_id, report_id)
# df_report_pages

In [ ]:
KUSTO_URI = "https://trd-q70msbdsfhsq3ntat4.z2.kusto.fabric.microsoft.com/"
DATABASE_NAME = "Monitoring Eventhouse"
TOTAL_HOURS = 3
TIME_RANGE = f'{TOTAL_HOURS}h'

kql_query = f"""
SemanticModelLogs
| where 1==1
    //and OperationDetailName == "DAXQuery"
    and WorkspaceName == "{workspace_to_get_reports}"
    //and ItemName == "People Tech Dataset"
    and Timestamp between (ago({TIME_RANGE}) .. ago(0h))
    and ExecutingUser == '96300124@gmodelo.com.mx'
| extend request_data = parse_json(ApplicationContext)
| extend dataset_id = tostring(request_data.DatasetId),
         report_id = tostring(request_data.Sources[0].ReportId),
         visual_id = tostring(request_data.Sources[0].VisualId),
         consumption_method = tostring(request_data.Sources[0].HostProperties.ConsumptionMethod),
         DAXQuery = EventText
// convert to UTC-3 before binning
| extend LocalTimestamp = datetime_add('hour', 0, Timestamp)
| extend Timestamp30s = bin(LocalTimestamp, 30s)
// summarize by detailed grouping
| summarize 
      TotalCpuTimeMs = sum(CpuTimeMs),
      TotalDurationMs = sum(DurationMs)
    by Timestamp30s, WorkspaceId, report_id, visual_id, ApplicationName, DatasetMode, OperationDetailName, ExecutingUser, DAXQuery
// compute totals per (Timestamp30s, WorkspaceId, ExecutingUser)
| join kind=leftouter (
    SemanticModelLogs
    | where 1==1
        and OperationDetailName == "DAXQuery"
        and WorkspaceName == "Brewdat Prod - GHQ - People - PSI"
        and ItemName == "People Tech Dataset"
        and Timestamp between (ago({TIME_RANGE}) .. ago(0h))
    | extend request_data = parse_json(ApplicationContext)
    | extend LocalTimestamp = datetime_add('hour', 0, Timestamp)
    | extend Timestamp30s = bin(LocalTimestamp, 30s)
    | summarize
          IntervalTotalCpuMs = sum(CpuTimeMs),
          IntervalTotalDurationMs = sum(DurationMs)
        by Timestamp30s, WorkspaceId, ExecutingUser
) on Timestamp30s, WorkspaceId, ExecutingUser
| project Timestamp30s, WorkspaceId, report_id, visual_id, ApplicationName, DatasetMode, OperationDetailName, ExecutingUser, DAXQuery,
          TotalCpuTimeMs, TotalDurationMs, IntervalTotalCpuMs, IntervalTotalDurationMs
| order by Timestamp30s asc
"""


df_kql = kqldb.query_kql_database(kql_query, sort_by='Timestamp30s')
df_kql.head()

In [ ]:
workspaces_and_reports = df_kql[['WorkspaceId', 'report_id']].drop_duplicates().reset_index(drop=True)
workspaces_and_reports = workspaces_and_reports[workspaces_and_reports['report_id']!='']
workspaces_and_reports_list = workspaces_and_reports.to_dict(orient='records')
workspaces_and_reports_list

In [ ]:
report_pages = []
for workspace_report in workspaces_and_reports_list:
    ws = workspace_report['WorkspaceId']
    rpt = workspace_report['report_id']
    print(f'Getting report pages and visuals for workspace {ws} and report {rpt}...')
    report_json = report.get_legacy_report_json(workspace_id=ws, report_id=rpt, operations=operations).get('content')
    df_report_page = report.get_legacy_report_pages_and_visuals(report_json, ws, rpt)
    report_pages.append(df_report_page)

df_report_pages = pd.concat(report_pages, ignore_index=True)

In [ ]:
columns_to_keep = ['Timestamp30s', 'WorkspaceName', 'report_name', 'pageName', 'visual_type', 'visual_title', 'OperationDetailName', 'ExecutingUser', 'DAXQuery', 'TotalDurationSec', 'TotalCUs']

df = pd.merge(df_kql, df_reports[['id', 'name']], how='left', left_on='report_id', right_on='id')
df['WorkspaceName'] = workspace_to_get_reports
df.drop(labels=['id'], axis='columns', inplace=True)
df = df[df['report_id']!='']
df.reset_index(drop=True, inplace=True)
df.rename(columns={'name': 'report_name'}, inplace=True)

df = pd.merge(df, df_report_pages[['report_id', 'visual_id', 'pageIndex', 'pageName', 'type', 'title']], how='left', on=['report_id', 'visual_id'])
df.reset_index(drop=True, inplace=True)
df.rename(columns={'type': 'visual_type', 'title': 'visual_title'}, inplace=True)

df['Timestamp30s'] = pd.to_datetime(df['Timestamp30s'])
df['TotalDurationMs'] = pd.to_numeric(df['TotalDurationMs'])
df['IntervalTotalDurationMs'] = pd.to_numeric(df['IntervalTotalDurationMs'])
df['IntervalTotalCpuMs'] = pd.to_numeric(df['IntervalTotalCpuMs'])
df['TotalCpuTimeMs'] = pd.to_numeric(df['TotalCpuTimeMs'])
df['IntervalCUs'] = df['IntervalTotalCpuMs'] / 1000
df['TotalCUs'] = df['TotalCpuTimeMs'] / 1000
df['IntervalDurationSec'] = df['IntervalTotalDurationMs'] / 1000
df['TotalDurationSec'] = df['TotalDurationMs'] / 1000
# df['%_capacity'] = df['TotalCUs'] / 1920
df['Timestamp30s'] = df['Timestamp30s'].dt.tz_convert('Etc/GMT+3')
df['Timestamp30s'] = df['Timestamp30s'].dt.strftime('%Y-%m-%d %H:%M:%S')
df = df[columns_to_keep]
df.sort_values(by=['Timestamp30s', 'TotalCUs'], inplace=True, ascending=[False, False])
df.to_excel(f"./data/monitoring/monitoring_{workspace_to_get_reports.replace(' ', '_').replace('-', '').replace('__', '_').strip()}_last_{TIME_RANGE}.xlsx", index=False)
df#['TotalCUs'].sum()

In [ ]:
reports_to_get_visuais = df['report_id'].unique()

### Test: change_data_destination (Lakehouse → Warehouse) + upgrade to CI/CD

In [ ]:
# Test parameters
test_workspace_id = '953072b3-e4c7-4551-8886-bbdca85faa2e'
test_dataflow_id = '506d7272-f0f3-4851-aed2-43ed47a6a3ed'  # Lakehouse Gen2 standard
test_warehouse_id = 'a40c0f15-2d35-4e95-9008-8b096007f577'

# Step 1: Get the standard Gen2 dataflow definition (lakehouse)
print("Step 1: Fetching standard Gen2 dataflow definition...")
pbi_result = dataflow._get_dataflow_pbi_definition(test_workspace_id, test_dataflow_id)
print(f"Result: {pbi_result['message']}")
pbi_content = pbi_result['content']
original_name = pbi_content.get('name', 'dataflow')
print(f"Dataflow name: {original_name}")

In [ ]:
# Step 2: Change data destination from Lakehouse to Warehouse
print("Step 2: Changing data destination from Lakehouse to Warehouse...")
modified_definition = dataflow.change_data_destination(
    definition=pbi_content,
    destination_type='Warehouse',
    destination_workspace_id=test_workspace_id,
    destination_item_id=test_warehouse_id
)

# Verify the change
doc = modified_definition['pbi:mashup']['document']
print(f"Has Fabric.Warehouse: {'Fabric.Warehouse' in doc}")
print(f"Has Lakehouse.Contents: {'Lakehouse.Contents' in doc}")
print(f"Has warehouseId: {'warehouseId' in doc}")

# Show connections
for conn in modified_definition['pbi:mashup'].get('connectionOverrides', []):
    print(f"  Connection: kind={conn['kind']}, path={conn['path']}")

In [ ]:
# Step 3: Convert modified definition to CI/CD format and create
new_display_name = f"{original_name}_cicd_dw_test"
print(f"Step 3: Converting to CI/CD and creating as '{new_display_name}'...")

cicd_definition = dataflow._convert_gen2_to_cicd_definition(
    modified_definition, new_display_name
)

# Verify CI/CD payload before creating
import base64
for part in cicd_definition['definition']['parts']:
    payload = base64.b64decode(part['payload']).decode('utf-8')
    if part['path'] == 'mashup.pq':
        print(f"\nmashup.pq preview (DataDestination query):")
        import re
        dd_match = re.search(r'shared \w+_DataDestination = let[\s\S]*?;\n', payload)
        if dd_match:
            print(dd_match.group(0))
    elif part['path'] == 'queryMetadata.json':
        meta = json.loads(payload)
        print(f"queryMetadata.json connections:")
        for conn in meta.get('connections', []):
            print(f"  kind={conn['kind']}, path={conn['path']}")

In [ ]:
# Step 4: Create the Dataflow Gen2 CI/CD with warehouse destination
print(f"Step 4: Creating Dataflow Gen2 CI/CD '{new_display_name}' in workspace {test_workspace_id}...")
create_result = dataflow.create_dataflow_gen2_from_definition(
    workspace_id=test_workspace_id,
    display_name=new_display_name,
    definition=cicd_definition
)
print(f"Result: {create_result['message']}")
if create_result.get('message') == 'Success':
    print(f"New dataflow ID: {create_result['content']['id']}")
else:
    print(f"Details: {create_result}")